# Lab: Teaching a Network to Judge Movie Reviews
### CSC-114 Artificial Intelligence I · Module 4 · IMDB Sentiment

**Companion to:** the Module 4 reading, *"Positive or Negative?"* — and to Chollet & Watson, *Deep Learning with Python*, 3rd ed., Ch. 4.

This lab is the **Practice** stage. Nothing here is graded. The goal is to *run the thing*, watch it overfit with your own eyes, and change one knob at a time to see what happens.

---

### Which track are you?

- **Prompt Masters** — Run every cell top to bottom. Read the outputs. For the experiments at the end, ask your AI partner to suggest a change, then make that one edit and re-run.
- **Code Builders** — Same, but you'll edit the experiment cells yourself and commit your results through the Sacred Flow (Issue → Branch → PR → Review → Merge).

Both tracks run the same notebook. The difference is how much you edit and how you submit.

---

### Before you run anything

- **First run downloads ~80 MB** (the IMDB dataset). Give it a minute. You only download it once.
- On **Google Colab**, the backend is already set — just run the cells.
- **Running locally?** Keras 3 works on JAX, TensorFlow, or PyTorch. The cell below picks one. If you hit a backend error, that's the line to change.

In [ ]:
# Pick a backend BEFORE importing keras. "jax" matches the textbook notebook.
# (On Colab this is already handled; running it again is harmless.)
import os
os.environ["KERAS_BACKEND"] = "jax"

import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers

print("Keras version:", keras.__version__)
print("Backend:", keras.backend.backend())
# CHECKPOINT: if this printed a version and a backend with no error, you're ready.

## Step 1 — Load the data

We keep only the **10,000 most common words**. Rare words get dropped — most appear in a single review and can't teach a general pattern.

In [ ]:
from keras.datasets import imdb

(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=10000)

print("Training reviews:", len(train_data))
print("Test reviews:    ", len(test_data))
print("First label (0=negative, 1=positive):", train_labels[0])
# What you should see: 25000 training, 25000 test, and a label of 0 or 1.

## Step 2 — Peek at one review

Each review is already a list of numbers, where every number stands for a word.
Because we capped the vocabulary at 10,000, no word-number goes above 9999.

In [ ]:
print("Review #0 as numbers (first 12):", train_data[0][:12])
print("Highest word-number anywhere:", max(max(seq) for seq in train_data))
# What you should see: a list of integers, and a max of 9999.

## Step 3 — Decode review #0 back into English

The numbers 0, 1, and 2 are **reserved** (padding, start-of-review, unknown word),
so every real word is shifted by 3. We undo that shift to read the review.

**Watch this detail** — forget the `- 3` shift and the decoded text comes out as nonsense.
(Your AI partner sometimes forgets it. Now *you* won't.)

In [ ]:
word_index = imdb.get_word_index()
reverse_word_index = dict((value, key) for (key, value) in word_index.items())
decoded_review = " ".join(reverse_word_index.get(i - 3, "?") for i in train_data[0])

print(decoded_review[:120])
# What you should see: a readable, clearly POSITIVE review starting with "?".
# The leading "?" is the start-of-review token, which has no English word.

## Step 4 — Turn each review into a row of numbers (multi-hot encoding)

Reviews have different lengths, but the network needs every input the same shape.
So we build a row of 10,000 slots — one per vocabulary word — and put a **1** in the
slot for every word the review contains, **0** everywhere else.

In [ ]:
def multi_hot_encode(sequences, num_classes):
    results = np.zeros((len(sequences), num_classes))
    for i, sequence in enumerate(sequences):
        results[i][sequence] = 1.0
    return results

x_train = multi_hot_encode(train_data, num_classes=10000)
x_test  = multi_hot_encode(test_data,  num_classes=10000)

# Labels just need to be the right number type.
y_train = train_labels.astype("float32")
y_test  = test_labels.astype("float32")

print("x_train shape:", x_train.shape)   # (25000, 10000): one row per review, 10000 wide
print("Values are only 0s and 1s. Max value:", x_train.max())
# CHECKPOINT: shape should be (25000, 10000) and the max should be 1.0.

## Step 5 — Build the model

Three layers. The first two find patterns; the last one gives a single yes/no answer.

- `Dense(16, "relu")` ×2 — modest room to learn non-straight-line patterns.
- `Dense(1, "sigmoid")` — **the binary-classification ending.** One number out, squashed to 0–1, read as *probability the review is positive*.

In [ ]:
model = keras.Sequential([
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1,  activation="sigmoid"),
])

## Step 6 — Compile (choose how it learns)

- **Loss = `binary_crossentropy`** — the right match for a probability output. It punishes confident wrong answers hard.
- **Optimizer = `adam`** — a reliable default.
- We also **monitor accuracy** (the plain % correct), but the model trains on the loss.

In [ ]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

## Step 7 — Hold out a validation set

We never tune against the **test** set — that would let the model peek at the final exam.
Instead we carve a **validation** set out of the training data to make decisions with
(like how long to train). The test set stays sealed until the very end.

In [ ]:
x_val = x_train[:10000]
partial_x_train = x_train[10000:]
y_val = y_train[:10000]
partial_y_train = y_train[10000:]

print("Train on:", partial_x_train.shape[0], "reviews | Validate on:", x_val.shape[0], "reviews")

## Step 8 — Train for 20 epochs and watch what happens

One **epoch** = one full pass over the training data. We'll do 20 and record how the model
does on both the training and validation sets at every step.

In [ ]:
history = model.fit(
    partial_x_train, partial_y_train,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val),
    verbose=2,
)
print("\nRecorded metrics:", list(history.history.keys()))

## Step 9 — Plot the curves (the most important picture in this unit)

Look for the moment the **validation** loss stops dropping and starts climbing.
That's where the model begins **overfitting** — memorizing the training reviews instead
of learning to read new ones.

In [ ]:
history_dict = history.history
epochs = range(1, len(history_dict["loss"]) + 1)

plt.plot(epochs, history_dict["loss"], "r--", label="Training loss")
plt.plot(epochs, history_dict["val_loss"], "b", label="Validation loss")
plt.title("[IMDB] Training and validation loss")
plt.xlabel("Epochs"); plt.ylabel("Loss"); plt.xticks(epochs); plt.legend()
plt.show()

In [ ]:
plt.clf()
plt.plot(epochs, history_dict["accuracy"], "r--", label="Training accuracy")
plt.plot(epochs, history_dict["val_accuracy"], "b", label="Validation accuracy")
plt.title("[IMDB] Training and validation accuracy")
plt.xlabel("Epochs"); plt.ylabel("Accuracy"); plt.xticks(epochs); plt.legend()
plt.show()

# What you should see: training keeps improving, but validation peaks early
# (often around epoch 4-5) and then gets WORSE. That gap is overfitting.
# WRITE DOWN the epoch where YOUR validation loss is lowest -- you'll need it next.

## Step 10 — Retrain for the right number of epochs, then grade once

Now we train a **fresh** model, but only for about **4 epochs** (stop where validation was best),
and evaluate on the sealed test set — one honest final grade.

> If your curve peaked at a different epoch, use that number instead of 4.

In [ ]:
model = keras.Sequential([
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1,  activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, epochs=4, batch_size=512, verbose=2)

results = model.evaluate(x_test, y_test, verbose=0)
print("\nTest loss: %.3f | Test accuracy: %.3f" % (results[0], results[1]))
# What you should see: roughly 0.88 accuracy. Your exact number will wobble a little
# each run because the model starts from random -- that's expected, not a bug.
# State-of-the-art methods reach about 0.95.

## Step 11 — Use the model on new reviews

`predict` returns a probability for each review: near **1** = confidently positive,
near **0** = confidently negative, near **0.5** = the model is genuinely unsure.

In [ ]:
preds = model.predict(x_test, verbose=0)
print("First 5 predictions:")
for p in preds[:5]:
    verdict = "positive" if p[0] >= 0.5 else "negative"
    print("  %.3f  -> %s" % (p[0], verdict))

## Step 12 — Experiments: change ONE thing at a time

This is the core discipline of the whole course: **change one knob, re-run, write down what happened.**
Don't change two things at once — you won't know which one caused the difference.

**Prompt Masters:** ask your AI partner *"which one of these should I try first, and what do you predict will happen?"* Then make that single edit below and run it. Compare its prediction to your result.

**Code Builders:** make the edits directly, and commit each experiment's result.

Try these, one per run, in a copy of the cell below:

1. Use **one** representation layer, or **three**, instead of two.
2. Use **32** or **64** units instead of 16.
3. Swap `binary_crossentropy` for `mean_squared_error`.
4. Swap `relu` for `tanh`.

Before each run, **write a one-line prediction.** Were you right?

In [ ]:
# EXPERIMENT TEMPLATE -- change exactly ONE line, then run.
# My prediction (fill in):  ____________________________________________

exp_model = keras.Sequential([
    layers.Dense(16, activation="relu"),   # <-- try 32 units? or tanh?
    layers.Dense(16, activation="relu"),   # <-- try deleting this line (one layer)?
    layers.Dense(1,  activation="sigmoid"),
])
exp_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",            # <-- try "mean_squared_error"?
    metrics=["accuracy"],
)
exp_history = exp_model.fit(
    partial_x_train, partial_y_train,
    epochs=20, batch_size=512,
    validation_data=(x_val, y_val), verbose=0,
)
best_val = max(exp_history.history["val_accuracy"])
print("Best validation accuracy this run: %.3f" % best_val)
# Compare to your baseline. What did your one change do?

## Record your experiments

Fill this in as you go (this is the kind of evidence your Apply/Assess submission will draw on):

| # | The one thing I changed | I predicted | Best val accuracy | What I learned |
|---|---|---|---|---|
| baseline | nothing | — | ~0.88 | starting point |
| 1 |  |  |  |  |
| 2 |  |  |  |  |
| 3 |  |  |  |  |

## Wrap-up — what to take from this lab

- Text has to become **numbers** before a network can use it (multi-hot encoding).
- A binary classifier **ends with one sigmoid unit** and trains with **`binary_crossentropy`**.
- More training is **not** automatically better — watch the **validation** curve and stop at the peak.
- One controlled change at a time is how you actually learn what matters.

**What to submit**
- *Prompt Masters:* your filled-in experiment table + a short note on one time your AI partner's prediction was wrong, and how you knew.
- *Code Builders:* the same, committed through the Sacred Flow, with your modified experiment cell(s) in the PR.

---
<!-- INSTRUCTOR NOTES — strip this cell before distribution if you prefer students not see expected numbers. For an ungraded Practice lab, leaving them in is usually helpful for mixed-ability learners. -->

```
INSTRUCTOR NOTES
================
VALIDATED: full pipeline runs clean on keras 3.14.1 / JAX backend (encode, build,
compile, fit w/ validation_data, evaluate, predict). Code is verbatim from Chollet's
MIT-licensed chapter04 notebook; only comments, checkpoints, and the experiment
scaffold were added.

EXPECTED OUTCOMES (will vary slightly by random init):
- Overfit point: validation loss bottoms out ~epoch 4-5, then rises.
- Final 4-epoch model: ~0.88 test accuracy. SOTA anchor ~0.95.
- predict() returns probabilities; confident samples near 0.99 / 0.01.

LIKELY EXPERIMENT RESULTS (so you can react in real time):
- 1 vs 3 layers: small effect; 3 layers may overfit slightly sooner.
- 32/64 units: trains a bit faster to fit, tends to overfit sooner; little test gain.
- mean_squared_error: still learns, usually a touch worse than crossentropy.
- tanh vs relu: comparable here; relu generally the safer default.

TRAP TIE-INS (reused in exit ticket + Apply/Assess):
- Step 3 seeds the decode offset-by-3 trap.
- Step 7 seeds the "don't tune on the test set" trap.
- Step 10's note seeds "more epochs != better."
- No softmax/categorical_crossentropy anywhere -- an agent suggesting them for this
  2-class problem is wrong (binary-vs-multiclass trap).

PLATFORM NOTES:
- imdb.load_data + get_word_index download on first run (~80 MB total). Verify lab
  network allows it, or pre-cache.
- Backend set to jax to match the textbook; TF/PyTorch produce equivalent results.
- Re-verify Keras dataset API + adam default each term.
```